This file takes the base data folder and pulls all metadata from every img, It also filters duplicates

In [1]:
import pandas as pd
import os
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import hashlib
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

/workspace/AML-3-MVL-AI-Classifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## parsing metadata into dataframe

In [40]:


DATA_PATH = Path('../data/raw').resolve()
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
NAME_MAP = {
    'adm': 'ADM',
    'glide': 'Glide',
    'midjourney': 'Midjourney',
    'sdv4': 'Stable Diffusion v1.4',
    'sdv5': 'Stable Diffusion v1.5',
    'vqdm': 'VQDM',
    'wukong': 'Wukong'
}
errors = []
image_records = []
file_paths = []
# make list of all files in subdirectories with the correct file extension
for root, dirs, files in os.walk(DATA_PATH):
    for file in files:
        if any(file.lower().endswith(ext) for ext in EXTENSIONS):
            file_paths.append(os.path.join(root, file))

print(f"{len(file_paths):,} images found")

# run through the list of files and extract metadata
for file_path in tqdm(file_paths, desc="parsing img"):
    try:
        file_size = os.path.getsize(file_path)
        
        with Image.open(file_path) as img:
            w, h = img.size
            path_obj = Path(file_path)
            folder_name = path_obj.parent.name
            model_type = NAME_MAP.get(folder_name, folder_name)
            parts = [p.lower() for p in path_obj.parts]
            category = 'ai' if 'ai' in parts else 'nature'
            
            image_records.append({
                'path': file_path,
                'filename': path_obj.name,
                'model_type': model_type,
                'category': category,
                'width': w,
                'height': h,
                'aspect_ratio': round(w / h, 2),
                'megapixels': round((w * h) / 1e6, 2),
                'size_bytes': file_size,
                'format': img.format
            })
    except Exception as e:
        # Keep track of what went wrong
        errors.append((file_path, str(e)))
        continue

    
print(f"Skipped {len(errors)} files due to errors.")

# Print the first 5 errors to diagnose the issue
for path, err in errors[:5]:
    print(f"File: {path} | Error: {err}")
df = pd.DataFrame(image_records)
df.to_parquet('image_metadata.parquet', engine='pyarrow', index=False)


2,345,167 images found


parsing img: 100%|██████████| 2345167/2345167 [08:50<00:00, 4423.96it/s]


Skipped 3661 files due to errors.
File: /workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_851.JPEG | Error: cannot identify image file '/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_851.JPEG'
File: /workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_8521.JPEG | Error: cannot identify image file '/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_8521.JPEG'
File: /workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_8525.JPEG | Error: cannot identify image file '/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_8525.JPEG'
File: /workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_8531.JPEG | Error: cannot identify image file '/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_8531.JPEG'
File: /workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034_8541.JPEG | Error: cannot identify image file '/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv5/n03444034

In [39]:
len(df)

2341506

In [23]:
# In case parquet is built, this line can read it into a dataframe object
df = pd.read_parquet('image_metadata.parquet')

In [ ]:
len(df)

## Parse Duplicates

Based on the metadata saved in the dataframe we can look at potential duplicate images based on same width, height and file size

In [8]:

dupe_mask = df.duplicated(subset=['size_bytes', 'width', 'height'], keep=False)
num_duplicates = dupe_mask.sum()

print(f"total images: {len(df):,}")
print(f"potential dupes: {num_duplicates:,} ({(num_duplicates/len(df)):.1%})")

if num_duplicates > 0:
    display(df[dupe_mask].sort_values(by='size_bytes').head(10))

print(df[dupe_mask]['category'].value_counts())


total images: 2,341,506
potential dupes: 1,224,912 (52.3%)


,path,filename,model_type,category,width,height,aspect_ratio,megapixels,size_bytes,format
1093082,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/wukong/[DUPE] n02089867_692.JPEG,[DUPE] n02089867_692.JPEG,Wukong,nature,50,33,1.52,0.00,1042,JPEG
851729,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/vqdm/n02089973_755.JPEG,n02089973_755.JPEG,VQDM,nature,50,33,1.52,0.00,1042,JPEG
92122,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/adm/n03467068_4892.JPEG,n03467068_4892.JPEG,ADM,nature,64,64,1.00,0.00,1149,JPEG
593174,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv4/n03467068_14786.JPEG,n03467068_14786.JPEG,Stable Diffusion v1.4,nature,64,64,1.00,0.00,1149,JPEG
431718,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/midjourney/[DUPE] n02504013_1017.JPEG,[DUPE] n02504013_1017.JPEG,Midjourney,nature,64,64,1.00,0.00,1188,JPEG
225866,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/glide/n02504458_1204.JPEG,n02504458_1204.JPEG,Glide,nature,64,64,1.00,0.00,1188,JPEG
575404,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv4/n02951585_12865.JPEG,n02951585_12865.JPEG,Stable Diffusion v1.4,nature,100,75,1.33,0.01,1222,JPEG
857834,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/vqdm/n02099429_1158.JPEG,n02099429_1158.JPEG,VQDM,nature,100,75,1.33,0.01,1222,JPEG
515729,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/sdv4/n01873310_12619.JPEG,n01873310_12619.JPEG,Stable Diffusion v1.4,nature,48,48,1.00,0.00,1329,JPEG
377621,/workspace/AML-3-MVL-AI-Classifier/data/raw/nature/midjourney/n02128385_1164.JPEG,n02128385_1164.JPEG,Midjourney,nature,48,48,1.00,0.00,1329,JPEG


category
ai        906029
nature    318883
Name: count, dtype: int64


On these potential dupes we run a simple script to calculate the file hash to check if the files are actually the same

In [9]:


dupe_indices = df[dupe_mask].index

print(f"Hashing {len(dupe_indices):,} potential duplicates...")

def calc_file_hash(file_path):
    hasher = hashlib.md5()
    try:
        with open(file_path, 'rb') as f:
            for chunk in iter(lambda: f.read(65536), b""):
                hasher.update(chunk)
        return hasher.hexdigest()
    except Exception:
        return None


tqdm.pandas(desc="hashing files")
df.loc[dupe_indices, 'file_hash'] = df.loc[dupe_indices, 'path'].progress_apply(calc_file_hash)
df['unique_id'] = df['file_hash'].fillna(df['path'])


Hashing 1,224,912 potential duplicates...


hashing files: 100%|██████████| 1224912/1224912 [14:59<00:00, 1361.13it/s]


In [10]:
df.to_parquet('image_hashed', engine='pyarrow', index=False)

In [ ]:
#makes a separate dataframe copy with only the files containing a hash
df_hashed = df[df['file_hash'].notna()].copy()

hash_dupe_mask = df_hashed.duplicated(subset=['file_hash'], keep=False)
all_duplicates = df_hashed[hash_dupe_mask].copy()
hash_dupe_only_mask = df_hashed.duplicated(subset=['file_hash'], keep='first')
only_duplicates = df_hashed[hash_dupe_only_mask].copy()
all_duplicates = all_duplicates.sort_values(by=['file_hash', 'category'])
ai_duplicates = all_duplicates[all_duplicates['category'] == 'ai']
# 4. Set display options for the full path

print(f"hashed {len(df_hashed):,} potential duplicates")
print(f"total duplicate files: {len(all_duplicates):,}")

print(f"of which are category 'ai': {len(ai_duplicates):,}")
display(ai_duplicates[['file_hash', 'category', 'path']].head(10))


hashed 1,224,912 potential duplicates
total duplicate files: 10,383
of which are category 'ai': 170


,file_hash,category,path
1997551,0636a7b6f209fe8e910cd35c4c3d45a4,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/midjourney/98_midjourney_80.png
1997554,0636a7b6f209fe8e910cd35c4c3d45a4,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/midjourney/[DUPE] 98_midjourney_84.png
1346799,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/116_sdv5_00035.png
1427516,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE] 116_sdv5_00059.png
1427517,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE] 116_sdv5_00064.png
1427518,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE] 116_sdv5_00097.png
1427519,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE] 116_sdv5_00149.png
1427520,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE] 176_sdv5_00131.png
1427521,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE] 263_sdv5_00055.png
1427522,10237f3f5141584b0d041790689803ec,ai,/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE] 263_sdv5_00057.png


In [ ]:
# adds first instance of stable diffusion safety filter to also tag that for removal
only_duplicates = pd.concat([only_duplicates, df[df['path'] == '/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/116_sdv5_00035.png']], ignore_index=True)

The occurrences found will be marked with a [DUPE] prefix for easy deletion later, also for the safety filter images also the first occurrence will be marked as [DUPE]

In [26]:
import os

rename_count = 0
skip_count = 0

for idx, row in tqdm(only_duplicates.iterrows(), total=len(only_duplicates), desc="renaming files"):
    old_path = Path(row['path'])
    
    # Skip if [DUPE] is already in the filename
    if old_path.name.startswith("[DUPE]"):
        skip_count += 1
        continue
    
    # Construct the new path
    new_name = f"[DUPE]{old_path.name}"
    new_path = old_path.with_name(new_name)
    
    try:
        os.rename(old_path, new_path)
        # Update the dataframe in case you use it later
        df.at[idx, 'path'] = str(new_path)
        df.at[idx, 'filename'] = new_name
        rename_count += 1
    except Exception as e:
        print(f"Error renaming {old_path.name}: {e}")

print(f"{rename_count} files marked. {skip_count} already had prefix.")

renaming files: 100%|██████████| 5313/5313 [00:00<00:00, 60408.18it/s]

Error renaming 116_sdv5_00035.png: [Errno 2] No such file or directory: '/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/116_sdv5_00035.png' -> '/workspace/AML-3-MVL-AI-Classifier/data/raw/ai/sdv5/[DUPE]116_sdv5_00035.png'
0 files marked. 5312 already had prefix.


## Parse Low Quality Images

After handling duplicate images we will take the original dataset and check for low quality images, this means a width or height that is lower than 256, since this will be the patch size of our model.

In [ ]:
lq_df = df[(df['width'] < 256) | (df['height'] < 256)]
print(f"{len(lq_df)} low quality images")
rename_count = 0
skip_count = 0

for idx, row in tqdm(lq_df.iterrows(), total=len(lq_df), desc="renaming files"):
    old_path = Path(row['path'])
    
    # Skip if [LQ] is already in the filename
    if old_path.name.startswith("[LQ]"):
        skip_count += 1
        continue
    
    # Construct the new path
    new_name = f"[LQ]{old_path.name}"
    new_path = original_path.parent / new_name
    
    try:
        os.rename(old_path, new_path)
        # Update the dataframe in case you use it later
        df.at[idx, 'path'] = str(new_path)
        df.at[idx, 'filename'] = new_name
        rename_count += 1
    except Exception as e:
        print(f"Error renaming {old_path.name}: {e}")

print(f"{rename_count} files marked. {skip_count} already had prefix.")

113197 low quality images


tagging low quality images:   0%|          | 0/113197 [00:00<?, ?it/s]

tagging low quality images:   3%|▎         | 3053/113197 [00:00<00:04, 26311.30it/s]

tagging low quality images: 100%|██████████| 113197/113197 [28:20<00:00, 66.57it/s] 
